# AIS Ship Trajectory Analysis with GeoPandas
This notebook queries AIS vessel positions from a SQLite database and visualizes ship trajectories on interactive maps using GeoPandas.

## 1. Set the MMSI of the ship you would like to work with:

In [17]:
# 338839000 MMSI of the ship that goes in the carribean
# 369970910 MMSI of the ship that goes around in the san diego bay
# 338812000 MMSI of the ship that goes chillin in a Jacksonville Florida bay
# 366998000 MMSI of the ship going to Mobile Alabama
# 369970968 MMSI of the another ship chillin in the Jacksonville Florida bay
# 368011000 MMSI of the ship going down the coast of California
# 369970707 MMSI of the ship leaving the cuban bay
# 368869000 MMSI of the ship that went to go chill in the Jacksonville bay

SHIP_MMSI = 369970707

## 2. Import Required Libraries

In [18]:
import sys
from pathlib import Path
import warnings

# Add actint to path for imports
actint_path = Path("/home/daxtonb/JFN-Groundtruth-Tools/actint/src")
if str(actint_path) not in sys.path:
    sys.path.insert(0, str(actint_path))

from actint.data_processing.query_database import query_ais_positions, query_vessels

warnings.filterwarnings('ignore')

## 3. Connect to Database and Query Ship Positions

In [19]:
# Query positions for a specific ship (configure as needed)
# For example: query_ais_positions({"MMSI": "123456789"}) or {"vessel_id": value}

SHIP_QUERY = {"MMSI": SHIP_MMSI}  # Change MMSI or column name as needed

try:
    ship_positions = query_ais_positions(SHIP_QUERY, sort=True)
    print(f"Retrieved {len(ship_positions)} positions")
    if ship_positions:
        print(f"Start position: {ship_positions[-1]}")
        print(f"End position: {ship_positions[0]}")
except Exception as e:
    print(f"Error querying database: {e}")
    ship_positions = []

2023-01-01T00:00:02.000000
Retrieved 960 positions
Start position: (3, 369970707, '2023-01-01T00:00:02.000000', 18.45901, -66.09607, 0.0, 104.4, 98.0, 'USS MILWAUKEE', None, 'NMLK', 90.0, 0.0, 107.0, None, None, 35.0, 'A', '2026-02-26 03:51:09')
End position: (9058, 369970707, '2023-01-01T19:11:31.000000', 18.6823, -66.02486, 11.6, 80.2, 81.0, 'USS MILWAUKEE', None, 'NMLK', 90.0, 0.0, 107.0, None, None, 35.0, 'A', '2026-02-26 03:51:09')


## 4 Interactive map with markers (ipyleaflet) 

### This might take a while to load depending on how  many points you have

In [20]:
from ipyleaflet import Map, Marker, AwesomeIcon, Popup, Rectangle
from ipywidgets import HTML
import math

# Example structure:
# ship_positions = [
#     [ship_id, timestamp, speed, lat, lon],
#     ...
# ]

middle_point = ship_positions[len(ship_positions)//2]

m = Map(center=(middle_point[3], middle_point[4]), zoom=10)

# first and last positions
last = ship_positions[0]
first = ship_positions[-1]

start_icon = AwesomeIcon(name="play", marker_color="green", icon_color="white")
end_icon = AwesomeIcon(name="flag", marker_color="red", icon_color="white")

def create_popup(position):
    ship_id = position[1]
    timestamp = position[2]
    lat = position[3]
    lon = position[4]

    html = HTML(f"""
    <b>Ship MMSI:</b> {ship_id}<br>
    <b>Timestamp:</b> {timestamp}<br>
    <b>Latitude:</b> {lat}<br>
    <b>Longitude:</b> {lon}<br>
    """)

    return Popup(child=html, close_button=True, auto_close=False)

# Add first and last markers
start_marker = Marker(location=(first[3], first[4]), icon=start_icon)
start_marker.popup = create_popup(first)
m.add(start_marker)

end_marker = Marker(location=(last[3], last[4]), icon=end_icon)
end_marker.popup = create_popup(last)
m.add(end_marker)

# Add a clickable marker every 50 positions
for i, position in enumerate(ship_positions[1:-1], start=1):
    if i % 100 == 0:
        icon = AwesomeIcon(name="circle", marker_color="blue", icon_color="white")
        marker = Marker(location=(position[3], position[4]), icon=icon)
        marker.popup = create_popup(position)
        m.add(marker)
    elif i%4 == 0:
        marker = Marker(location=(position[3], position[4]))
        m.add(marker)

m

Map(center=[18.45899, -66.09603], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', …

## 5. Filter and Analyze Multiple Ship Trajectories

## Points used to determine where the ship is going

In [21]:
NUMBER_DETECTIONS=300

middle_point = ship_positions[300]

m = Map(center=(middle_point[3], middle_point[4]), zoom=10)

# first and last positions
first = ship_positions[NUMBER_DETECTIONS-1]
last = ship_positions[0]

start_icon = AwesomeIcon(name="play", marker_color="green", icon_color="white")
end_icon = AwesomeIcon(name="flag", marker_color="red", icon_color="white")

# Add first and last markers
start_marker = Marker(location=(first[3], first[4]), icon=start_icon)
start_marker.popup = create_popup(first)
m.add(start_marker)

end_marker = Marker(location=(last[3], last[4]), icon=end_icon)
end_marker.popup = create_popup(last)
m.add(end_marker)

# Add a clickable marker every 50 positions
print(len(ship_positions[1:NUMBER_DETECTIONS-1]))
for i, position in enumerate(ship_positions[1:NUMBER_DETECTIONS-1], start=1):
    if i % 100 == 0:
        icon = AwesomeIcon(name="circle", marker_color="blue", icon_color="white")
        marker = Marker(location=(position[3], position[4]), icon=icon)
        marker.popup = create_popup(position)
        m.add(marker)
    else:
        marker = Marker(location=(position[3], position[4]))
        m.add(marker)

m

298


Map(center=[18.45898, -66.09601], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', …

## A couple of helper functions

In [22]:
def vectorize(lat1, lon1, lat2, lon2):
    lat = lat2-lat1
    lon = lon2-lon1
    return (lat, lon)


def within_angle(a, b, tolerance=15):
    diff = abs((a - b + 180) % 360 - 180)
    return diff <= tolerance

## Code to determine what direction the ship is gonig

In [23]:
VECTOR_DISTANCE_RATIO = 0.9
DEGREE_THRESHOLD= 15
 #This is the number of detections used to calculate where a ship is going.
from actint.tools.utils.important_locations import *
from actint.data_processing.query_database import query_ais_positions
from datetime import datetime, timedelta
from actint.tools.utils.distance_calculation import calculate_bearing, haversine_distance_nm
from actint.tools.lat_lon_context import identify_maritime_region


def calculate_vector_and_distance_sum(ship_mmsi: str, number_detections=300, tracking_time=timedelta(hours=1)):
    positions = query_ais_positions({"mmsi": ship_mmsi}, sort=True)
    
    position1 = positions[number_detections-1]        #This is 300 or whatever the number_detections is away from the most rescent position
    print(position1)

    rescent_reversed_positions = reversed(positions[0:number_detections-1])
    total_vector = [0.0, 0.0]
    total_distance = 0

    for position2 in rescent_reversed_positions:
        print(position2[3], position2[4], position2[2])
        latlng = vectorize(position1[3], position1[4], position2[3], position2[4])
        total_vector[0] += latlng[0]
        total_vector[1] += latlng[1]
        total_distance += math.hypot(latlng[0], latlng[1])
        position1 = position2
    
    

    vector_distance_ratio = math.hypot(total_vector[0], total_vector[1])/total_distance
    print("Vector distance ratio", vector_distance_ratio)
    if(vector_distance_ratio > VECTOR_DISTANCE_RATIO):                                                       #Can use this to describe if the ship is going fast or slow
        print("The ship is going toward something")
        return (total_vector, total_distance)
    else:
        print("The ship is doing wierd stuff acting like a reet.")
        return (total_vector, total_distance)
    

        
(total_vector, total_distance) = calculate_vector_and_distance_sum(SHIP_MMSI, 300)

print(total_vector)




2023-01-01T00:00:02.000000
(6007, 369970707, '2023-01-01T12:56:02.000000', 18.45898, -66.09601, 0.0, 63.6, 98.0, 'USS MILWAUKEE', None, 'NMLK', 90.0, 0.0, 107.0, None, None, 35.0, 'A', '2026-02-26 03:51:09')
18.45899 -66.09601 2023-01-01T12:57:11.000000
18.45898 -66.09601 2023-01-01T12:58:21.000000
18.45899 -66.09601 2023-01-01T12:59:31.000000
18.45897 -66.096 2023-01-01T13:00:02.000000
18.45898 -66.09601 2023-01-01T13:01:51.000000
18.45898 -66.09602 2023-01-01T13:03:02.000000
18.45898 -66.09601 2023-01-01T13:04:12.000000
18.45897 -66.09602 2023-01-01T13:05:21.000000
18.45897 -66.09602 2023-01-01T13:06:32.000000
18.45897 -66.096 2023-01-01T13:07:41.000000
18.45897 -66.09601 2023-01-01T13:08:52.000000
18.45896 -66.09601 2023-01-01T13:10:02.000000
18.45898 -66.09601 2023-01-01T13:11:12.000000
18.45897 -66.09598 2023-01-01T13:12:21.000000
18.45897 -66.09601 2023-01-01T13:13:31.000000
18.45897 -66.096 2023-01-01T13:14:41.000000
18.45897 -66.09601 2023-01-01T13:15:52.000000
18.45899 -66.096

In [24]:
current_position = ship_positions[0]

print(current_position)
m = Map(center=(current_position[3], current_position[4]), zoom=10)

# Add first and last markers
current_pos_marker = Marker(location=(current_position[3], current_position[4]))
current_pos_marker.popup = create_popup(current_position)
m.add(current_pos_marker)

(dy, dx) = total_vector



start_lat = current_position[3]
start_lon = current_position[4]

end_lat = start_lat + dy
end_lon = start_lon + dx

from ipyleaflet import Polyline

vector_line = Polyline(
    locations=[(start_lat, start_lon), (end_lat, end_lon)],
    color="red",
    weight=3
)

m.add(vector_line)


(9058, 369970707, '2023-01-01T19:11:31.000000', 18.6823, -66.02486, 11.6, 80.2, 81.0, 'USS MILWAUKEE', None, 'NMLK', 90.0, 0.0, 107.0, None, None, 35.0, 'A', '2026-02-26 03:51:09')


Map(center=[18.6823, -66.02486], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', '…

## Figure out the potential destinations the ships could be going to.

In [25]:
def create_bounding_box_on_map(map_obj, bounds, color='blue', fill_opacity=0.2, weight=3):
    """Add a rectangular bounding box to an ipyleaflet Map.

    bounds should be a dict-like object with keys:
      lat_min, lat_max, lon_min, lon_max
    """
    required_keys = {'lat_min', 'lat_max', 'lon_min', 'lon_max'}
    if not required_keys.issubset(bounds):
        raise ValueError(f"bounds must include {required_keys}")

    lat_min = float(bounds['lat_min'])
    lat_max = float(bounds['lat_max'])
    lon_min = float(bounds['lon_min'])
    lon_max = float(bounds['lon_max'])

    if lat_min > lat_max:
        lat_min, lat_max = lat_max, lat_min

    rectangles = []
    # Handle simple non-date-line bounding boxes
    if lon_min <= lon_max:
        rectangles.append((lat_min, lon_min, lat_max, lon_max))
    else:
        # Crossing the dateline, split into two rectangles
        rectangles.append((lat_min, lon_min, lat_max, 180))
        rectangles.append((lat_min, -180, lat_max, lon_max))

    added = []
    for rlat_min, rlon_min, rlat_max, rlon_max in rectangles:
        rectangle = Rectangle(
            bounds=[
                (rlat_min, rlon_min),
                (rlat_min, rlon_max),
                (rlat_max, rlon_max),
                (rlat_max, rlon_min),
            ],
            color=color,
            weight=weight,
            fill_color=color,
            fill_opacity=fill_opacity,
        )
        map_obj.add_layer(rectangle)
        added.append(rectangle)

    if len(added) == 1:
        return added[0]
    return added

for key, value in MARITIME_REGIONS.items():
    print(value['bounds'])
    create_bounding_box_on_map(m, value['bounds'])


for key, value in CONTINENTS.items():
    print(value['bounds'])
    create_bounding_box_on_map(m, value['bounds'])
    
for key, value in OCEANS.items():
    print(value['bounds'])
    create_bounding_box_on_map(m, value['bounds'])

m

{'lat_min': 18, 'lat_max': 31, 'lon_min': -98, 'lon_max': -80}
{'lat_min': 9, 'lat_max': 22, 'lon_min': -88, 'lon_max': -60}
{'lat_min': 30, 'lat_max': 46, 'lon_min': -6, 'lon_max': 36}
{'lat_min': 0, 'lat_max': 23, 'lon_min': 100, 'lon_max': 121}
{'lat_min': 23, 'lat_max': 33, 'lon_min': 120, 'lon_max': 130}
{'lat_min': 33, 'lat_max': 52, 'lon_min': 127, 'lon_max': 142}
{'lat_min': 5, 'lat_max': 35, 'lon_min': 120, 'lon_max': 145}
{'lat_min': 5, 'lat_max': 25, 'lon_min': 50, 'lon_max': 75}
{'lat_min': 24, 'lat_max': 30, 'lon_min': 48, 'lon_max': 56}
{'lat_min': 12, 'lat_max': 30, 'lon_min': 32, 'lon_max': 44}
{'lat_min': -60, 'lat_max': 30, 'lon_min': 20, 'lon_max': 100}
{'lat_min': 52, 'lat_max': 66, 'lon_min': 162, 'lon_max': 180}
{'lat_min': 52, 'lat_max': 66, 'lon_min': -180, 'lon_max': -157}
{'lat_min': 10, 'lat_max': 90, 'lon_min': -135, 'lon_max': -50}
{'lat_min': -56, 'lat_max': 14, 'lon_min': -85, 'lon_max': -34}
{'lat_min': 32, 'lat_max': 75, 'lon_min': -13, 'lon_max': 46}
{

Map(center=[18.6823, -66.02486], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', '…

In [26]:
def get_possible_destinations(current_position, direction_vector):

    current_lat = current_position[0]
    current_lon = current_position[1]

    #Find the nearest port or waterway that the ship is going toward
    nearest_potential_waterway_name = None
    nearest_potential_waterway = [999999, 999999]
    for name, location in STRATEGIC_WATERWAYS.items():
        location_degrees = calculate_bearing(current_lat, current_lon, location[0], location[1])
        direction_degrees = math.degrees(math.atan2(direction_vector[0], direction_vector[1]))
        if(within_angle(location_degrees, direction_degrees, DEGREE_THRESHOLD)):
            location_vector = vectorize(current_lat, current_lon, location[0], location[1])
            if(math.hypot(nearest_potential_waterway[0], nearest_potential_waterway[1]) > math.hypot(location_vector[0], location_vector[1])):
                nearest_potential_waterway = location_vector
                nearest_potential_waterway_name = name

    print(nearest_potential_waterway_name)
    
    
    nearest_potential_port_name = None    
    nearest_potential_port = [999999, 999999]
    for name, location in MAJOR_PORTS.items():
        location_degrees = calculate_bearing(current_lat, current_lon, location[0], location[1])
        direction_degrees = math.degrees(math.atan2(direction_vector[0], direction_vector[1]))
        if(within_angle(location_degrees, direction_degrees, DEGREE_THRESHOLD)):
            location_vector = vectorize(current_lat, current_lon, location[0], location[1])
            if(math.hypot(nearest_potential_port[0], nearest_potential_port[1]) > math.hypot(location_vector[0], location_vector[1])):
                nearest_potential_port = location_vector
                nearest_potential_port_name = name

    if math.hypot(nearest_potential_waterway[0], nearest_potential_waterway[1]) < math.hypot(nearest_potential_port[0], nearest_potential_port[1]):
        nearest_thing = nearest_potential_waterway
        nearest_thing_name = nearest_potential_waterway_name
    else:
        nearest_thing = nearest_potential_port
        nearest_thing_name = nearest_potential_port_name

    max_fov_continent_name = None
    max_fov_maritime_name = None

    print(haversine_distance_nm(current_position[0], current_position[1], nearest_thing[0], nearest_thing[1]) )
    if haversine_distance_nm(current_position[0], current_position[1], nearest_thing[0], nearest_thing[1]) < 300:
        return f"The ship is going toward {nearest_thing_name}"

    else:

        for continent_name, bounding_box in CONTINENTS.items():
            
            
            lat_min = bounding_box['lat_min']
            lat_max = bounding_box['lat_max']
            lon_min = bounding_box['lon_min']
            lon_max = bounding_box['lon_max']

            current_lat = current_position[0]
            current_lon = current_position[1]

            
            degree1 = calculate_bearing(current_lat, current_lon, lat_min, lon_min)
            degree2 = calculate_bearing(current_lat, current_lon, lat_min, lon_max)
            degree3 = calculate_bearing(current_lat, current_lon, lat_max, lon_min)
            degree4 = calculate_bearing(current_lat, current_lon, lat_max, lon_max)  #get the degrees of all the bearings, then get the largest and smallest ones

            max_degree = max(degree1, degree2, degree3, degree4)
            min_degree = min(degree1, degree2, degree3, degree4)

            #subtract the bearing of the ship, and cut the two bearings off at the at a max +-DEGREE_THRESHOLD
            max_fov_degree = 0
            if(min_degree < DEGREE_THRESHOLD and max_degree > -1*DEGREE_THRESHOLD):
                
                fov_min_degree = max(min_degree, -1*DEGREE_THRESHOLD)
                fov_max_degree = min(max_degree, DEGREE_THRESHOLD)

                total_fov_degrees = fov_max_degree - fov_min_degree
                if total_fov_degrees > max_fov_degree:
                    max_fov_degree = total_fov_degrees
                    max_fov_continent_name = continent_name
                
                
        
        for maritime_name, bounding_box in MARITIME_REGIONS.items():
            
            maritime_name = identify_maritime_region(current_lat, current_lon)
            if continent_name == maritime_name:
                continue
            
            lat_min = bounding_box['bounds']['lat_min']
            lat_max = bounding_box['bounds']['lat_max']
            lon_min = bounding_box['bounds']['lon_min']
            lon_max = bounding_box['bounds']['lon_max']

            current_lat = current_position[0]
            current_lon = current_position[1]

            degree1 = calculate_bearing(current_lat, current_lon, lat_min, lon_min)
            degree2 = calculate_bearing(current_lat, current_lon, lat_min, lon_max)
            degree3 = calculate_bearing(current_lat, current_lon, lat_max, lon_min)
            degree4 = calculate_bearing(current_lat, current_lon, lat_max, lon_max)  #get the degrees of all the bearings, then get the largest and smallest ones

            max_degree = max(degree1, degree2, degree3, degree4)
            min_degree = min(degree1, degree2, degree3, degree4)

            #subtract the bearing of the ship, and cut the two bearings off at the at a max +-DEGREE_THRESHOLD
            max_fov_degree = 0
            if(min_degree < DEGREE_THRESHOLD and max_degree > -1*DEGREE_THRESHOLD):
                
                fov_min_degree = max(min_degree, -1*DEGREE_THRESHOLD)
                fov_max_degree = min(max_degree, DEGREE_THRESHOLD)

                total_fov_degrees = fov_max_degree - fov_min_degree
                if total_fov_degrees > max_fov_degree:
                    max_fov_degree = total_fov_degrees
                    max_fov_maritime_name = continent_name

            #store these values



        #get the max fov continent and ocean (aside from the ocean the boat is in)

        #The ship is going through {closest_ocean} to {closest_continent}

        
    return f"The ship is in going toward {maritime_name} to {max_fov_continent_name}"

current_position = (current_position[3], current_position[4])
get_possible_destinations(current_position, total_vector)

Strait of Gibraltar
6922.319649857724


KeyError: 'lat_min'